# TARGET ratio T50 — 70/30 và 60/40
Chỉ chạy trên máy từ xa. Chọn CPU, không cần GPU/TPU. Upload gói TARGET_RATIO_T50_70_30_60_40.zip bằng Add Input (Kaggle, dataset private) hoặc upload vào /content (Colab). Đọc README trong gói. Chưa có kết quả cho hai tỷ lệ mới.
Hai máy chọn hai tỷ lệ khác nhau; mỗi phần30lượt. Giữ mọi seed/kết quả. Checkpoint theo lượt hoàn tất, không theo vòng; disk tạm có thể mất khi VM bị thu hồi.

In [ ]:
import os, sys
from pathlib import Path
print(sys.version)
visible = os.cpu_count() or 1
affinity = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else visible
effective = min(visible, affinity)
quota = Path('/sys/fs/cgroup/cpu.max')
if quota.exists():
    q, period = quota.read_text().split()
    if q != 'max': effective = min(effective, max(1, int(q)//int(period)))
print('CPU visible:', visible, 'affinity:', affinity, 'effective workers:', effective)
if sys.version_info < (3,11): raise RuntimeError('Python >=3.11 required; prefer3.12')

In [ ]:
from zipfile import ZipFile
import hashlib, subprocess
home = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
candidates = list(Path('/kaggle/input').rglob('TARGET_RATIO_T50*.zip')) if Path('/kaggle/input').exists() else list(home.glob('TARGET_RATIO_T50*.zip'))
if len(candidates) != 1: raise RuntimeError('Upload exactly one study ZIP; adjust candidates if needed')
archive = candidates[0]
print('ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
project = home/'target_ratio_package'
if not project.exists():
    project.mkdir()
    with ZipFile(archive) as z:
        for member in z.infolist():
            target = (project/member.filename).resolve()
            if not target.is_relative_to(project.resolve()): raise ValueError('Unsafe ZIP path')
        z.extractall(project)
subprocess.run([sys.executable,'-m','pip','install','numpy','matplotlib','psutil'],check=True)
launcher = project/'Core Algrithms/tools/run_target_ratio_t50.py'
subprocess.run([sys.executable,str(launcher),'check'],cwd=project,check=True)

## Chọn phần và nơi lưu bền vững trước khi chạy
Điền GA70_PO30 hoặc GA60_PO40 vào RATIO. Đặt OUTPUT trên nơi bạn kiểm soát và sao lưu. Nếu thư mục đã chuẩn bị bằng prepare thì đặt RESUME=True. Trên Colab có thể mount Drive theo hướng dẫn của Google và đặt OUTPUT trên Drive; nhiều file nhỏ có thể chậm. Không cho hai phiên dùng cùng OUTPUT. Không có bảo đảm phiên free chạy đủ12giờ.

In [ ]:
RATIO = 'SET_ME'  # GA70_PO30 or GA60_PO40
OUTPUT = home/('RATIO_T50_'+RATIO)  # change to persistent storage if available
RESUME = False
if RATIO not in ('GA70_PO30','GA60_PO40'): raise ValueError('Choose one ratio before starting')
command = [sys.executable,'-u',str(launcher),'run','--ratio',RATIO,'--output-root',str(OUTPUT),'--workers',str(effective),'--acknowledge-compute']
if RESUME: command.append('--resume')
subprocess.run(command,cwd=project,check=True)

In [ ]:
import json, shutil
status = json.loads((OUTPUT/'status.json').read_text())
print(status)
if status['status'] != 'COMPLETED': raise RuntimeError('Preserve the complete output directory; do not treat an incomplete part as final')
result_zip = shutil.make_archive(str(home/OUTPUT.name),'zip',root_dir=OUTPUT.parent,base_dir=OUTPUT.name)
print('Download and keep the entire result archive:', result_zip)